# ANOVA Analysis Template (One-Way & Factorial)

**Reusable Jupyter notebook for multi-group mean comparisons following the workflow in Denis (2021), Chapter 6.**

This template is designed to be adapted to any continuous outcome and categorical factor(s).  
Replace the example data with your own, update variable names, and re-run.

---

### Contents
1. Setup & Data Loading  
2. Exploratory Data Analysis  
3. Assumption Checks  
4. One-Way ANOVA  
5. Effect Size  
6. Post-Hoc Tests (Tukey HSD)  
7. Factorial ANOVA & Interaction  
8. Residual Diagnostics  
9. Reporting Template  


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.graphics.gofplots import qqplot

# Display settings
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
%matplotlib inline

print("Libraries loaded successfully.")


## 2. Data Loading

**Example data** (teacher achievement scores from Denis, 2021).  
Replace this block with your own data-loading code.


In [ ]:
# --- EXAMPLE DATA (replace with your own) ---
data = {
    'ac': [70, 67, 65, 75, 76, 73,   # Teacher 1
           69, 68, 70, 76, 77, 75,   # Teacher 2
           85, 86, 85, 76, 75, 73,   # Teacher 3
           95, 94, 89, 94, 93, 91],  # Teacher 4
    'teach': [1]*6 + [2]*6 + [3]*6 + [4]*6,
    'text':  [1,1,1,2,2,2] * 4       # Textbook factor for factorial demo
}
df = pd.DataFrame(data)

# Ensure factors are categorical
df['teach'] = df['teach'].astype('category')
df['text']  = df['text'].astype('category')

print(df.head(10))
print("\nShape:", df.shape)
print("\nGroup sizes:\n", df.groupby('teach', observed=True).size())


## 3. Exploratory Data Analysis

In [ ]:
# Descriptive statistics by group
print("=== Means & SDs by Teacher ===")
print(df.groupby('teach', observed=True)['ac'].agg(['count', 'mean', 'std', 'min', 'max']))

print("\n=== Overall ===")
print(df['ac'].describe())


In [ ]:
# Visual: boxplots
fig, ax = plt.subplots()
sns.boxplot(data=df, x='teach', y='ac', ax=ax, palette='Set2')
ax.set_title('Achievement by Teacher')
ax.set_xlabel('Teacher')
ax.set_ylabel('Achievement Score')
plt.show()


In [ ]:
# Histograms by group
df['ac'].hist(by=df['teach'], bins=5, figsize=(10, 6), layout=(2, 2))
plt.suptitle('Histograms of Achievement by Teacher')
plt.tight_layout()
plt.show()


## 4. Assumption Checks

ANOVA assumes:
1. **Independence** of observations (design-based; not tested statistically here).
2. **Normality** of residuals (or of the DV within groups).
3. **Homogeneity of variance** across groups.


In [ ]:
# Shapiro-Wilk on the entire DV (guide only; better per group with larger n)
stat, p = stats.shapiro(df['ac'])
print(f"Shapiro-Wilk (overall): W = {stat:.4f}, p = {p:.4f}")
print("→ Small p suggests departure from normality; interpret with caution for small n.")


In [ ]:
# Levene's test for homogeneity of variance
# Note: for one-way we need the groups as separate arrays or use formula interface
groups = [group['ac'].values for name, group in df.groupby('teach', observed=True)]
stat, p = stats.levene(*groups)
print(f"Levene's test: statistic = {stat:.4f}, p = {p:.4f}")
if p < 0.05:
    print("→ Evidence of unequal variances. ANOVA is reasonably robust, but note it.")
else:
    print("→ No strong evidence against equal variances.")


## 5. One-Way ANOVA

In [ ]:
# Fit the model
model = ols('ac ~ C(teach)', data=df).fit()

# ANOVA table (Type II SS)
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)


In [ ]:
# Manual extraction for clarity
ss_between = anova_table.loc['C(teach)', 'sum_sq']
ss_resid   = anova_table.loc['Residual', 'sum_sq']
df_between = anova_table.loc['C(teach)', 'df']
df_resid   = anova_table.loc['Residual', 'df']
ms_between = ss_between / df_between
ms_resid   = ss_resid / df_resid
F_stat     = ms_between / ms_resid
p_value    = anova_table.loc['C(teach)', 'PR(>F)']

print(f"SS Between = {ss_between:.3f}")
print(f"SS Residual = {ss_resid:.3f}")
print(f"MS Between = {ms_between:.3f}")
print(f"MS Residual = {ms_resid:.3f}")
print(f"F({df_between:.0f}, {df_resid:.0f}) = {F_stat:.3f}, p = {p_value:.2e}")


## 6. Effect Size (η²)

In [ ]:
# Eta-squared
eta_sq = ss_between / (ss_between + ss_resid)
print(f"η² = {eta_sq:.3f}")
print(f"→ Approximately {eta_sq*100:.1f}% of the variance in achievement is attributable to teacher differences.")

# Optional: partial omega-squared (less biased)
# ω² ≈ (SS_b - df_b * MS_e) / (SS_total + MS_e)
omega_sq = (ss_between - df_between * ms_resid) / (ss_between + ss_resid + ms_resid)
print(f"ω² (approx) = {omega_sq:.3f}")


## 7. Post-Hoc Tests (Tukey HSD)

In [ ]:
# Tukey Honestly Significant Difference
tukey = pairwise_tukeyhsd(endog=df['ac'], groups=df['teach'], alpha=0.05)
print(tukey)


In [ ]:
# Visual summary of pairwise comparisons (optional)
tukey.plot_simultaneous()
plt.title('Tukey HSD Simultaneous Confidence Intervals')
plt.show()


## 8. Factorial ANOVA (Two-Way) & Interaction

When a second factor is available, include it and the interaction term.


In [ ]:
# Two-way model WITH interaction
model_2way = ols('ac ~ C(teach) * C(text)', data=df).fit()
anova_2way = sm.stats.anova_lm(model_2way, typ=2)
print(anova_2way)


In [ ]:
# Interaction plot
interaction = df.groupby(['teach', 'text'], observed=True)['ac'].mean().unstack()
interaction.plot(marker='o', linewidth=2)
plt.title('Interaction Plot: Achievement by Teacher × Textbook')
plt.xlabel('Teacher')
plt.ylabel('Mean Achievement')
plt.legend(title='Textbook')
plt.grid(True, alpha=0.3)
plt.show()


**Interpretation tip:**  
Always interpret the **interaction first**. If the interaction is significant, the effect of one factor depends on the level of the other; main-effect statements alone are incomplete.


## 9. Residual Diagnostics

In [ ]:
# Residuals from the two-way model (or one-way if preferred)
resid = model_2way.resid

# QQ plot
fig = sm.qqplot(resid, line='s')
plt.title('QQ Plot of Residuals')
plt.show()


In [ ]:
# Residual vs fitted
fitted = model_2way.fittedvalues
plt.scatter(fitted, resid, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted')
plt.show()


## 10. Reporting Template (copy & adapt)

**Results paragraph example (one-way):**

> A one-way ANOVA was conducted to examine the effect of teacher on student achievement scores.  
> There was a statistically significant difference among the four teachers,  
> *F*(3, 20) = 31.21, *p* < .001, η² = .82.  
> Post-hoc Tukey HSD tests indicated that Teacher 4 differed significantly from Teachers 1, 2, and 3;  
> Teacher 3 also differed from Teachers 1 and 2. Teachers 1 and 2 did not differ significantly from each other.

**Always include:**
- Exact *F*, degrees of freedom, and *p*-value (or *p* < .001)
- Effect size (η² or ω²)
- Direction of differences (which groups are higher/lower)
- Brief statement on assumptions or robustness


---

## How to Reuse This Template

1. Replace the data dictionary / `pd.read_csv(...)` block with your own data.  
2. Rename columns consistently (`dv`, `factor1`, `factor2`, …).  
3. Update the `ols` formulas.  
4. Re-run assumption checks and interpret them in the new scientific context.  
5. Adjust the reporting language to match your domain.

**Key principle:** The model you fit determines the effects you see. Always document the full model specification.
